In [1]:
!pip install -q ultralytics
import yaml, shutil, pandas as pd, numpy as np
from pathlib import Path
from ultralytics import YOLO
inp, W = Path("/kaggle/input"), Path("/kaggle/working")

models = {}
for p in inp.rglob("best.pt"):
    name = p.parent.parent.name
    models["A_bd" if name == "nlm_y11s" else name] = p
print("Models found:", sorted(models))

src = {"bd": next(p for p in inp.rglob("nlm_tiles") if p.is_dir()),
       "gh": next(p for p in inp.rglob("gh_split") if p.is_dir()),
       "tz": next(p for p in inp.rglob("tz_split") if p.is_dir())}
for site, d in src.items():
    shutil.copytree(d / "test", W / f"{site}_test" / "test", dirs_exist_ok=True)
    yaml.safe_dump({"path": str(W / f"{site}_test"), "train": "test/images", "val": "test/images", "test": "test/images",
                    "names": {0: "parasite", 1: "wbc"}}, open(W / f"{site}_test.yaml", "w"))
    print(site, "test tiles:", len(list((W / f"{site}_test" / "test" / "images").glob("*.jpg"))))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Models found: ['A_bd', 'B_gh', 'C_tz', 'D_gh_tz', 'E_bd_tz', 'F_bd_gh']
bd test tiles: 2845
gh test tiles: 3943
tz test tiles: 1040


In [2]:
rows = []
for mname in sorted(models):
    model = YOLO(str(models[mname]))
    for site in ["bd", "gh", "tz"]:
        m = model.val(data=str(W / f"{site}_test.yaml"), split="test", imgsz=640, batch=32, device=0,
                      plots=False, verbose=False)
        rows.append(dict(model=mname, test_site=site, par_mAP50=m.box.ap50[0], par_recall=m.box.r[0],
                         par_precision=m.box.p[0], wbc_mAP50=m.box.ap50[1]))
res = pd.DataFrame(rows)
res.to_csv(W / "cross_site_detection.csv", index=False)

print("PARASITE mAP50 (rows = model, columns = country it is tested on)\n")
print(res.pivot(index="model", columns="test_site", values="par_mAP50").round(3))
print("\nWBC mAP50\n")
print(res.pivot(index="model", columns="test_site", values="wbc_mAP50").round(3))

Ultralytics 8.4.191 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 751.5±217.3 MB/s, size: 46.3 KB)
val: Scanning /kaggle/working/bd_test/test/labels... 2845 images, 27 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2845/2845 1.2Kit/s 2.3s0.1s
val: New cache created: /kaggle/working/bd_test/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 89/89 2.4it/s 36.7s0.4ss
                   all       2845      37096      0.924      0.906      0.938      0.496
Speed: 0.8ms preprocess, 9.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Ultralytics 8.4.191 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1162.8±368.3 MB/s, siz